<a href="https://colab.research.google.com/github/dhenis09/Analysis_Case_Project/blob/main/Ocean_Engines_Shipping_Commercial_Performance_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**🚢 Ocean Engines Shipping — Commercial Performance Analysis**

## Business Context

Ocean Engines Shipping's Commercial Team is exploring opportunities to improve vessel commercialization and maximize fleet profitability. With voyage-level commercial and operational data available, the team aims to identify the key factors influencing voyage performance and uncover opportunities to optimize vessel deployment.

This analysis will examine the relationship between vessel characteristics, commercial routes, utilization, freight rates, voyage duration, and operating costs to support data-driven commercial decisions.

## Business Objectives

This project aims to answer three key business questions:

1. **Voyage Performance**
   - Which voyages demonstrate the strongest commercial performance?
   - Which voyages have the lowest profitability?
   - What operational and commercial characteristics distinguish high-performing and low-performing voyages?

2. **Route Commercialization**
   - Which routes are most profitable for each vessel type?
   - How should vessels be commercially deployed across different routes to maximize profitability?
   - Are there specific vessel types that perform better on particular routes?

3. **Bunker & Cost Optimization**
   - How significant is bunker cost across total voyage costs?
   - How does bunker cost vary by vessel type and voyage characteristics?
   - What opportunities exist to optimize bunker-related costs while maintaining commercial performance?

## Analytical Approach

The analysis will combine:

- **SQL** for data exploration, transformation, aggregation, and business analysis
- **Python** for exploratory data analysis and statistical investigation
- **Power BI** for interactive visualization and management-oriented storytelling

The ultimate goal is to translate voyage-level data into **actionable commercial recommendations** that can support vessel deployment, route selection, and cost optimization decisions.

## Data Preparation

In [ ]:
!pip install duckdb

In [ ]:
import pandas as pd
import numpy as np
import duckdb
from scipy import stats

In [ ]:
#Load data
com_data = pd.read_csv('/content/Ocean_Engine_shipping_commercial_performance.csv',delimiter = ",")

In [ ]:
com_data.head()

,voyage_id,voyage_date,vessel,vessel_size,charter_type,business_segment,origin_region,destination_region,distance_nm,cargo_volume_mt,...,utilization_rate,freight_rate_usd_mt,revenue_usd,bunker_cost_usd,port_cost_usd,operating_cost_usd,other_cost_usd,total_cost_usd,profit_usd,profit_margin
0,VY-0369,2025-01-01,GA-05,Midsize,Owned,Import,Middle East,Southeast Asia,1660,38258,...,0.989002,45.216941,1710.884839,9287.427946,30734.300516,121414.846186,129.418375,161565.993023,-159855.108185,-93.434172
1,VY-0713,2025-01-01,GA-03,Small II,Owned,Charter Out,East Asia,Southeast Asia,6471,35556,...,0.946328,22.389277,753.346655,23242.921473,31410.287503,171259.284883,47.641048,225960.134907,-225206.788252,-298.941778
2,VY-0520,2025-01-02,GA-10,VLGC,Spot,Import,Domestic,Domestic,4620,57632,...,0.822539,59.367412,2814.284887,40715.703454,49160.203309,289134.674513,245.641133,379256.222409,-376441.937522,-133.761134
3,VY-0516,2025-01-02,GA-07,VLGC,Owned,Import,Domestic,Domestic,8410,52310,...,0.899605,60.823236,2862.239599,61745.316957,48568.572798,337925.252190,186.563630,448425.705576,-445563.465976,-155.669521
4,VY-0806,2025-01-02,GA-04,Small II,Time Charter,Import,Middle East,Southeast Asia,539,27684,...,0.987720,32.218890,880.994678,2020.933004,25103.475797,24428.654647,46.373034,51599.436483,-50718.441805,-57.569521


In [ ]:
#Data checking
com_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 21 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   voyage_id            1200 non-null   object 
 1   voyage_date          1200 non-null   object 
 2   vessel               1200 non-null   object 
 3   vessel_size          1200 non-null   object 
 4   charter_type         1200 non-null   object 
 5   business_segment     1200 non-null   object 
 6   origin_region        1200 non-null   object 
 7   destination_region   1200 non-null   object 
 8   distance_nm          1200 non-null   int64  
 9   cargo_volume_mt      1200 non-null   int64  
 10  voyage_days          1200 non-null   float64
 11  utilization_rate     1200 non-null   float64
 12  freight_rate_usd_mt  1200 non-null   float64
 13  revenue_usd          1200 non-null   float64
 14  bunker_cost_usd      1200 non-null   float64
 15  port_cost_usd        1200 non-null   f

Data type telah sesuai

In [ ]:
#Checking for anomalies
com_data.describe(include='all')

,voyage_id,voyage_date,vessel,vessel_size,charter_type,business_segment,origin_region,destination_region,distance_nm,cargo_volume_mt,...,utilization_rate,freight_rate_usd_mt,revenue_usd,bunker_cost_usd,port_cost_usd,operating_cost_usd,other_cost_usd,total_cost_usd,profit_usd,profit_margin
count,1200,1200,1200,1200,1200,1200,1200,1200,1200.000000,1200.000000,...,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000
unique,1200,453,10,4,3,3,5,4,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,VY-0929,2025-04-18,GA-06,Midsize,Time Charter,Import,Middle East,Southeast Asia,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,9,140,375,495,560,487,459,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5351.425833,38831.487500,...,0.818569,46.438115,1615.898401,31146.698306,31655.626479,204012.873118,105.835691,266921.033594,-265305.135193,-219.101558
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2973.806559,13069.687884,...,0.096595,16.068531,1007.950121,21059.763967,12232.273828,120653.753732,72.459131,145263.980551,144728.965874,162.535463
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,252.000000,8000.000000,...,0.485245,18.000000,153.261498,929.681624,5000.000000,11654.749362,9.193541,29125.562463,-742763.461590,-1352.768025
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2807.750000,27735.000000,...,0.755543,33.089230,747.432135,13821.973400,22122.080297,108212.240438,45.411451,151946.316103,-359305.862230,-271.804815
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5349.000000,38606.500000,...,0.822342,46.043898,1411.120032,26950.915143,30130.211176,181694.078447,88.500481,238958.264064,-236591.835570,-171.519184
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7920.750000,49866.750000,...,0.884825,59.309538,2382.959877,44878.307761,39459.363653,279127.718229,152.644813,360965.629059,-150495.703179,-111.860846


Kolom yang berisi bilangan telah sesuai, tidak ditemukan data anomali

In [ ]:
#Duplicate id checking
com_data['voyage_id'].duplicated().sum()

np.int64(0)

Tidak ada data duplikat

In [ ]:
#Checking date format
date_check = pd.to_datetime(com_data['voyage_date'], format='%Y-%m-%d', errors='coerce') #Change voyage_date into date format

#Count invalid date for
invalid_date = date_check.isna() | (date_check.dt.year >2026)

print(invalid_date.sum(), ' invalid values')

0  invalid values


Format tanggal telah sesuai

In [ ]:
#Check perhitungan pnl
invalid_profit = com_data[
      ~np.isclose(
          com_data['revenue_usd'],
          com_data['freight_rate_usd_mt'] * com_data['cargo_volume_mt'] * com_data['utilization_rate'])]
print(invalid_profit['revenue_usd'].sum(), ' invalid values')

1939078.0808078093  invalid values


## Data Processing

In [ ]:
#Calling SQL
con = duckdb.connect()

In [ ]:
##SQL Testing
con.sql("""
SELECT *
FROM 'Ocean_Engine_shipping_commercial_performance.csv'
LIMIT 10;
""")

1. Voyage Performance

Which voyages demonstrate the strongest commercial performance? Commercial performance doesn't always mean bigger revenue, it also need to have low maintenance/cost.

To See strongest commercial performance, we will pick top 5 vessel by PnL

In [ ]:
con.sql("""
SELECT profit_usd, profit_margin
FROM 'Ocean_Engine_shipping_commercial_performance.csv'
ORDER BY profit_usd DESC
LIMIT 5;
""")